# CHALLENGE 2: DAX to SQL Foundations

Reinforce the core SQL patterns you just learned in Demo 5. Each task maps directly to a DAX concept you already know.

**Instructions:**
- Replace the `<FILL_IN>` placeholder(s) with the correct SQL
- Run each cell to verify your answer
- Solutions are at the bottom — try first!

**Tables available:**
- `quarterly_sales` — 384 rows (12 months × 4 regions × 4 categories × 2 channels)
- `category_targets` — 4 rows (annual revenue target per category)

---

In [0]:
%run ./challenge_2_setup

## Task 1: Total Revenue by Region (SUM + GROUP BY)

Calculate the **total revenue** and **total units sold** for each region.

**DAX equivalent:** `SUMMARIZE(Sales, Sales[region], "Revenue", SUM(Sales[revenue]))`

**Expected result:** 4 rows, one per region. North should have the highest revenue.

In [0]:
%sql
-- Task 1: Replace <FILL_IN> with the correct aggregate function and clause
SELECT 
  region,
  ROUND(<FILL_IN>(revenue), 2) AS total_revenue,
  SUM(units_sold) AS total_units
FROM quarterly_sales
<FILL_IN> region
ORDER BY total_revenue DESC;

---

## Task 2: Filter Then Aggregate (WHERE + GROUP BY)

Find the total revenue for **only online sales** (`channel = 'online'`), grouped by product category.

**DAX equivalent:** `CALCULATE(SUM(Sales[revenue]), Sales[channel] = "online")`

**Expected result:** 4 rows. Online revenue should be higher than in-store for every category.

In [0]:
%sql
-- Task 2: Replace <FILL_IN> with the correct filter
SELECT 
  product_category,
  ROUND(SUM(revenue), 2) AS online_revenue
FROM quarterly_sales
WHERE <FILL_IN>
GROUP BY product_category
ORDER BY online_revenue DESC;

---

## Task 3: Prior Month Comparison (LAG)

Calculate **monthly total revenue** and add a column showing the **prior month’s revenue** using `LAG()`.

**DAX equivalent:** `CALCULATE([Total Revenue], DATEADD(Dates[Date], -1, MONTH))`

**Hint:** `LAG(expression)` looks back exactly one row in the ordered window.

**Expected result:** 12 rows. The first row’s prior month will be NULL.

In [0]:
%sql
-- Task 3: Replace <FILL_IN> with the window function that looks back one row
SELECT 
  report_month,
  ROUND(SUM(revenue), 2) AS monthly_revenue,
  ROUND(<FILL_IN>(SUM(revenue)) OVER (ORDER BY report_month), 2) AS prior_month_revenue
FROM quarterly_sales
GROUP BY report_month
ORDER BY report_month;

---

## Task 4: Classify with CASE WHEN

Group by `region` and classify each region’s total revenue as:
- `'Strong'` if total revenue >= 250000
- `'Moderate'` if total revenue >= 200000
- `'Needs Growth'` otherwise

**DAX equivalent:** `SWITCH(TRUE(), [Total Revenue] >= 250000, "Strong", ...)`

**Expected result:** 4 rows. Each region gets a performance label.

In [0]:
%sql
-- Task 4: Replace <FILL_IN> with the full CASE WHEN expression
SELECT 
  region,
  ROUND(SUM(revenue), 2) AS total_revenue,
  <FILL_IN> AS performance
FROM quarterly_sales
GROUP BY region
ORDER BY total_revenue DESC;

---

## Task 5: Percent of Total (Window Function with Empty OVER)

For each product category, show its revenue as a **percentage of the grand total**.

**DAX equivalent:** `DIVIDE([Category Revenue], CALCULATE([Total Revenue], ALL(Sales)))`

**Hint:** `SUM(...) OVER ()` with an empty `OVER()` gives the total across all rows.

**Expected result:** 4 rows. Percentages should add up to 100%.

In [0]:
%sql
-- Task 5: Replace <FILL_IN> with the window expression for the grand total
SELECT 
  product_category,
  ROUND(SUM(revenue), 2) AS category_revenue,
  ROUND(SUM(revenue) / <FILL_IN> * 100, 1) AS pct_of_total
FROM quarterly_sales
GROUP BY product_category
ORDER BY category_revenue DESC;

---

## Task 6: JOIN to Compare Actual vs Target

Join `quarterly_sales` to `category_targets` and calculate what **percentage of the annual target** each category has achieved.

**DAX equivalent:** Using a relationship between a fact table and a target table, then `DIVIDE([Actual], [Target])`.

**Hint:** Join on `product_category`.

**Expected result:** 4 rows with actual revenue, target, and % achieved.

In [0]:
%sql
-- Task 6: Replace <FILL_IN> with the JOIN type, condition, and target column
SELECT 
  q.product_category,
  ROUND(SUM(q.revenue), 2) AS actual_revenue,
  t.annual_target,
  ROUND(SUM(q.revenue) / t.<FILL_IN> * 100, 1) AS pct_achieved
FROM quarterly_sales q
<FILL_IN> category_targets t 
  ON q.product_category = t.product_category
GROUP BY q.product_category, t.annual_target
ORDER BY pct_achieved DESC;

---
---
---

## ⬇️ Solutions (Try the tasks first!) ⬇️

Scroll down only after you’ve attempted all tasks above.

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### Solution 1

Replace with `SUM` and `GROUP BY`.

In [0]:
%sql
SELECT 
  region,
  ROUND(SUM(revenue), 2) AS total_revenue,
  SUM(units_sold) AS total_units
FROM quarterly_sales
GROUP BY region
ORDER BY total_revenue DESC;

### Solution 2

Replace with `channel = 'online'`.

In [0]:
%sql
SELECT 
  product_category,
  ROUND(SUM(revenue), 2) AS online_revenue
FROM quarterly_sales
WHERE channel = 'online'
GROUP BY product_category
ORDER BY online_revenue DESC;

### Solution 3

Replace with `LAG` — it looks back one row in the ordered result.

In [0]:
%sql
SELECT 
  report_month,
  ROUND(SUM(revenue), 2) AS monthly_revenue,
  ROUND(LAG(SUM(revenue)) OVER (ORDER BY report_month), 2) AS prior_month_revenue
FROM quarterly_sales
GROUP BY report_month
ORDER BY report_month;

### Solution 4

Replace with the full CASE WHEN block. North and East are ‘Strong’, South is ‘Moderate’, West ‘Needs Growth’.

In [0]:
%sql
SELECT 
  region,
  ROUND(SUM(revenue), 2) AS total_revenue,
  CASE 
    WHEN SUM(revenue) >= 250000 THEN 'Strong'
    WHEN SUM(revenue) >= 200000 THEN 'Moderate'
    ELSE 'Needs Growth'
  END AS performance
FROM quarterly_sales
GROUP BY region
ORDER BY total_revenue DESC;

### Solution 5

Replace with `SUM(SUM(revenue)) OVER ()` — the outer `SUM() OVER ()` computes the grand total across all grouped rows.

In [0]:
%sql
SELECT 
  product_category,
  ROUND(SUM(revenue), 2) AS category_revenue,
  ROUND(SUM(revenue) / SUM(SUM(revenue)) OVER () * 100, 1) AS pct_of_total
FROM quarterly_sales
GROUP BY product_category
ORDER BY category_revenue DESC;

### Solution 6

Replace with `annual_target` (the column name) and `INNER JOIN` (the join type).

In [0]:
%sql
SELECT 
  q.product_category,
  ROUND(SUM(q.revenue), 2) AS actual_revenue,
  t.annual_target,
  ROUND(SUM(q.revenue) / t.annual_target * 100, 1) AS pct_achieved
FROM quarterly_sales q
INNER JOIN category_targets t 
  ON q.product_category = t.product_category
GROUP BY q.product_category, t.annual_target
ORDER BY pct_achieved DESC;